In [2]:
import numpy as np
import pickle
import json

In [4]:
train_data = np.load('../data/processed/train_data.npz')
val_data = np.load('../data/processed/val_data.npz')
test_data = np.load('../data/processed/test_data.npz')

In [5]:
# Extract Proccessed Data
X_train, y_train = train_data['X'], train_data['y']
X_val, y_val = val_data['X'], val_data['y']
X_test, y_test = test_data['X'], test_data['y']

In [6]:
X_train.shape, y_train.shape, X_val.shape, y_val.shape, X_test.shape, y_test.shape

((168, 224, 224, 3),
 (168,),
 (36, 224, 224, 3),
 (36,),
 (36, 224, 224, 3),
 (36,))

In [7]:
# Get number of classes
with open('../data/processed/label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)
num_classes= len(label_encoder.classes_)
print(f'Number of classes: {num_classes}')

Number of classes: 24


In [8]:
label_encoder.classes_

array(['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N',
       'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y'], dtype='<U1')

In [9]:
import sys
import os

# Add parent directory to path so we can import src module
sys.path.insert(0, os.path.abspath('..'))

In [10]:
# Build and train model
from src.model import SignLanguageModel_ResNet50
from src.training import TrainingPipeline

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 24)             │         3,096 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,148,248 (92.12 MB)

 Trainable params: 20,013,464 (76.35 MB)

 Non-trainable params: 4,134,784 (15.77 MB)

None


In [11]:
# Build Model
model_builder = SignLanguageModel_ResNet50(num_classes)
model = model_builder.build_model()
model = model_builder.compile_model(model)

In [12]:
print(model.loss)

sparse_categorical_crossentropy


In [13]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 24)             │         3,096 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,148,248 (92.12 MB)

 Trainable params: 20,013,464 (76.35 MB)

 Non-trainable params: 4,134,784 (15.77 MB)

In [14]:
# Train Model
config = {
    'batch_size': 32,
    'epochs': 2
}
trainer = TrainingPipeline(model, config)
history = trainer.train(X_train, y_train, X_val, y_val, '../models/best_model.h5')


Epoch 1/2
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.0181 - loss: 3.3849
Epoch 1: val_accuracy improved from None to 0.02778, saving model to ../models/best_model.h5



Epoch 1: finished saving model to ../models/best_model.h5
6/6 ━━━━━━━━━━━━━━━━━━━━ 96s 12s/step - accuracy: 0.0238 - loss: 3.3629 - val_accuracy: 0.0278 - val_loss: 3.5553 - learning_rate: 0.0010
Epoch 2/2
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.0839 - loss: 3.2349
Epoch 2: val_accuracy improved from 0.02778 to 0.05556, saving model to ../models/best_model.h5



Epoch 2: finished saving model to ../models/best_model.h5
6/6 ━━━━━━━━━━━━━━━━━━━━ 37s 6s/step - accuracy: 0.0893 - loss: 3.3207 - val_accuracy: 0.0556 - val_loss: 24.3243 - learning_rate: 0.0010
Restoring model weights from the end of the best epoch: 1.
